# 5CharCNN Training Loop Testbed

## Imports

In [ ]:
# --- Standard library ---
from pathlib import Path
import sys
import json
from datetime import datetime

# Ensure the repository root is on sys.path for local imports
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repository root found at: {REPO_ROOT}")

# --- Third-party ---
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# --- Local imports ---
from src.dataset.dataloader import create_dataloaders
from src.models.FiveCharCNN import FiveCharCaptchaCNN

## Configure Settings

In [ ]:
# =========================
# Config
# =========================

# Define path for saving runs and outputs
EXPERIMENTS_DIR = REPO_ROOT / "experiments"

# Create a new folder
RUN_NAME = f"test_run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = EXPERIMENTS_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Define paths for saving models, history, and plots
BEST_MODEL_PATH = RUN_DIR / "best_model.pt"
LAST_MODEL_PATH = RUN_DIR / "last_model.pt"
HISTORY_PATH = RUN_DIR / "training_history.json"
CONFIG_PATH = RUN_DIR / "config.json"
CURVES_PATH = RUN_DIR / "training_curves.png"


# =========================
# Core task config
# =========================

NUM_CHAR_CLASSES = 51
LABEL_LENGTH = 5


# =========================
# Training config
# =========================

BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 3e-4


# =========================
# Dataset split config
# =========================

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RANDOM_SEED = 42
TRAINING = True
SHUFFLE_TRAIN = True

NUM_WORKERS = 0
PIN_MEMORY = False
DROP_LAST = False

SUBSET_FRACTION = 1.0
RETURN_FILENAMES = False


# =========================
# Debug & Performance config
# =========================

SAVE_PLOTS = True
PRINT_BATCH_SHAPES = True


# =========================
# Device config
# =========================

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")
print(f"Run directory: {RUN_DIR}")

## Save Run Configurations

In [ ]:
CONFIG = {
    "num_char_classes": NUM_CHAR_CLASSES,
    "label_length": LABEL_LENGTH,
    
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "random_seed": RANDOM_SEED,
    
    "shuffle_train": SHUFFLE_TRAIN,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
    "drop_last": DROP_LAST,
    
    "subset_fraction": SUBSET_FRACTION,
}


with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=4)

## Create Data loaders

In [ ]:
# =========================
# Create dataloaders
# =========================

# Create train/validation/test dataloaders and character mappings
train_loader, val_loader, test_loader, char_to_idx, idx_to_char = create_dataloaders(
    batch_size=BATCH_SIZE,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    random_seed=RANDOM_SEED,
    training=TRAINING,
    shuffle_train=SHUFFLE_TRAIN,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=DROP_LAST,
    subset_fraction=SUBSET_FRACTION,
    return_filenames=RETURN_FILENAMES,
)

## Initialize model

In [ ]:
# =========================
# Initialize model
# =========================

#Initialize model and move to selecteddevice
model = FiveCharCaptchaCNN(
    num_char_classes=NUM_CHAR_CLASSES,
    label_length=LABEL_LENGTH
).to(DEVICE)

print(model)

criterion = nn.CrossEntropyLoss()

In [ ]:
# =========================
# Optimizer + history
# =========================

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {
    "train_loss": [],
    "val_loss": [],
    "train_char_acc": [],
    "val_char_acc": [],
    "train_seq_acc": [],
    "val_seq_acc": [],
    "val_pos_acc_0": [],
    "val_pos_acc_1": [],
    "val_pos_acc_2": [],
    "val_pos_acc_3": [],
    "val_pos_acc_4": [],
}

print("Optimizer, criterion, and history initialized.")

## Move to Python Script (engine)

In [ ]:
# =========================
# Metric helpers
# =========================

def compute_metrics(outputs, labels):
    """
    outputs: [B, 5, 51]
    labels:  [B, 5]
    """
    preds = outputs.argmax(dim=-1)  # [B, 5]

    char_acc = (preds == labels).float().mean().item()
    seq_acc = (preds == labels).all(dim=1).float().mean().item()

    pos_accs = []
    for pos in range(labels.size(1)):
        pos_acc = (preds[:, pos] == labels[:, pos]).float().mean().item()
        pos_accs.append(pos_acc)

    return char_acc, seq_acc, pos_accs

In [ ]:
# =========================
# Safe batch unpacking (if return_filenames is True)
# =========================
def unpack_batch(batch):
    if isinstance(batch, (list, tuple)):
        if len(batch) == 2:
            images, labels = batch
            filenames = None
        elif len(batch) == 3:
            images, labels, filenames = batch
        else:
            raise ValueError(f"Unexpected batch length: {len(batch)}")
    else:
        raise TypeError(f"Unexpected batch type: {type(batch)}")

    return images, labels, filenames

In [ ]:
# =========================
# Train for one epoch
# =========================

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    running_char_acc = 0.0
    running_seq_acc = 0.0
    num_batches = 0

    for batch in loader:
        images, labels, filenames = unpack_batch(batch)
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)  # [B, 5, 51]
        loss = criterion(outputs.view(-1, NUM_CHAR_CLASSES), labels.view(-1))

        loss.backward()
        optimizer.step()

        char_acc, seq_acc, _ = compute_metrics(outputs, labels)

        running_loss += loss.item()
        running_char_acc += char_acc
        running_seq_acc += seq_acc
        num_batches += 1

    epoch_loss = running_loss / num_batches
    epoch_char_acc = running_char_acc / num_batches
    epoch_seq_acc = running_seq_acc / num_batches

    return epoch_loss, epoch_char_acc, epoch_seq_acc

In [ ]:
# =========================
# Validate for one epoch
# =========================

def validate_one_epoch(model, loader, criterion, device, num_char_classes, label_length):
    model.eval()

    running_loss = 0.0
    running_char_acc = 0.0
    running_seq_acc = 0.0
    running_pos_accs = [0.0] * label_length
    num_batches = 0

    with torch.no_grad():
        for batch in loader:
            images, labels, _ = unpack_batch(batch)

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs.view(-1, num_char_classes), labels.view(-1))

            char_acc, seq_acc, pos_accs = compute_metrics(outputs, labels)

            running_loss += loss.item()
            running_char_acc += char_acc
            running_seq_acc += seq_acc

            for i in range(label_length):
                running_pos_accs[i] += pos_accs[i]

            num_batches += 1

    epoch_loss = running_loss / num_batches
    epoch_char_acc = running_char_acc / num_batches
    epoch_seq_acc = running_seq_acc / num_batches
    epoch_pos_accs = [x / num_batches for x in running_pos_accs]

    return epoch_loss, epoch_char_acc, epoch_seq_acc, epoch_pos_accs

## Training Loop

In [ ]:
# =========================
# Full training loop
# =========================

best_val_seq_acc = -1.0

for epoch in range(NUM_EPOCHS):
    train_loss, train_char_acc, train_seq_acc = train_one_epoch(
    model=model,
    loader=train_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    num_char_classes=NUM_CHAR_CLASSES,
)

    val_loss, val_char_acc, val_seq_acc, val_pos_accs = validate_one_epoch(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=DEVICE,
    num_char_classes=NUM_CHAR_CLASSES,
    label_length=LABEL_LENGTH,
)

    # Save metrics
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_char_acc"].append(train_char_acc)
    history["val_char_acc"].append(val_char_acc)
    history["train_seq_acc"].append(train_seq_acc)
    history["val_seq_acc"].append(val_seq_acc)

    for i in range(LABEL_LENGTH):
        history[f"val_pos_acc_{i}"].append(val_pos_accs[i])

    # Epoch panel
    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"Train Char Acc: {train_char_acc:.4f} | Val Char Acc: {val_char_acc:.4f}")
    print(f"Train Seq Acc:  {train_seq_acc:.4f} | Val Seq Acc:  {val_seq_acc:.4f}")
    print("Val Pos Accs:", [f"{acc:.4f}" for acc in val_pos_accs])

    # Save best model
    if val_seq_acc > best_val_seq_acc:
        best_val_seq_acc = val_seq_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"New best model saved to: {BEST_MODEL_PATH}")

# Save last model
torch.save(model.state_dict(), LAST_MODEL_PATH)
print(f"\nLast model saved to: {LAST_MODEL_PATH}")

In [ ]:
# =========================
# Save training history
# =========================

with open(HISTORY_PATH, "w") as f:
    json.dump(history, f, indent=4)

print(f"Training history saved to: {HISTORY_PATH}")

## Plot Curves


In [ ]:
# =========================
# Plot training curves
# =========================

epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
if SAVE_PLOTS:
    plt.savefig(CURVES_PATH.with_name("loss_" + CURVES_PATH.name), bbox_inches="tight")
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_char_acc"], label="Train Char Acc")
plt.plot(epochs, history["val_char_acc"], label="Val Char Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Character Accuracy")
plt.legend()
plt.grid(True)
if SAVE_PLOTS:
    plt.savefig(CURVES_PATH.with_name("char_acc_" + CURVES_PATH.name), bbox_inches="tight")
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_seq_acc"], label="Train Seq Acc")
plt.plot(epochs, history["val_seq_acc"], label="Val Seq Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Sequence Accuracy")
plt.legend()
plt.grid(True)
if SAVE_PLOTS:
    plt.savefig(CURVES_PATH.with_name("seq_acc_" + CURVES_PATH.name), bbox_inches="tight")
plt.show()


plt.figure(figsize=(8, 5))
for i in range(LABEL_LENGTH):
    plt.plot(epochs, history[f"val_pos_acc_{i}"], label=f"Val Pos {i}")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Per-Position Accuracy")
plt.legend()
plt.grid(True)
if SAVE_PLOTS:
    plt.savefig(CURVES_PATH.with_name("val_pos_acc_" + CURVES_PATH.name), bbox_inches="tight")
plt.show()

## Inspect Predictions

In [ ]:
# =========================
# Inspect validation predictions
# =========================

model.eval()

num_examples_to_show = 8

with torch.no_grad():
    batch = next(iter(val_loader))
    images, labels, filenames = unpack_batch(batch)

    images = images.to(DEVICE)
    labels = labels.to(DEVICE)

    outputs = model(images)
    preds = outputs.argmax(dim=-1)

for i in range(min(num_examples_to_show, images.size(0))):
    true_string = "".join(idx_to_char[int(x)] for x in labels[i])
    pred_string = "".join(idx_to_char[int(x)] for x in preds[i])

    print(f"Example {i+1}")
    print(f"True: {true_string}")
    print(f"Pred: {pred_string}")
    if filenames is not None:
        print(f"File: {filenames[i]}")
    print("-" * 30)

## Optional manual visual inspection

In [ ]:
# =========================
# Show validation images with true/pred labels
# =========================

num_images_to_show = 5

for i in range(min(num_images_to_show, images.size(0))):
    img_np = images[i].detach().cpu().squeeze().numpy()
    true_string = "".join(idx_to_char[int(x)] for x in labels[i])
    pred_string = "".join(idx_to_char[int(x)] for x in preds[i])

    plt.figure(figsize=(4, 2))
    plt.imshow(img_np, cmap="gray")
    plt.title(f"True: {true_string} | Pred: {pred_string}")
    plt.axis("off")
    plt.show()

## Debugging Graveyard

Cells below are diagnostic tools used when investigating model behavior


In [ ]:
# =========================
# Inspect one batch
# =========================

batch = next(iter(train_loader))

if isinstance(batch, (list, tuple)):
    if len(batch) == 2:
        images, labels = batch
        filenames = None
    elif len(batch) == 3:
        images, labels, filenames = batch
    else:
        raise ValueError(f"Unexpected batch length: {len(batch)}")
else:
    raise TypeError(f"Unexpected batch type: {type(batch)}")

if PRINT_BATCH_SHAPES:
    print("\nBatch inspection:")
    print(f"images.shape = {images.shape}")
    print(f"labels.shape = {labels.shape}")
    if filenames is not None:
        print(f"Example filename = {filenames[0]}")

print(f"images.dtype = {images.dtype}")
print(f"labels.dtype = {labels.dtype}")
print(f"labels[0] = {labels[0]}")

### Batch Visualization

In [ ]:
# =========================
# Decode and visualize n examples from the batch
# =========================
num_examples_to_show = 5

for i in range(num_examples_to_show):
    img = images[i]
    label = labels[i]
    
    decoded = "".join(idx_to_char[int(c)] for c in label)
    img_np = img.squeeze().cpu().numpy()
    
    plt.figure(figsize=(4,2))
    plt.imshow(img_np, cmap="gray")
    plt.title(decoded)
    plt.axis("off")
    plt.show()

### Foward pass test

In [ ]:

# Move batch to device
images = images.to(DEVICE)
labels = labels.to(DEVICE)

# Forward pass
outputs = model(images)

print("\nForward pass results:")
print(f"Input shape:  {images.shape}")
print(f"Output shape: {outputs.shape}")

### Loss test

In [ ]:
# =========================
# Loss test
# =========================

criterion = nn.CrossEntropyLoss()

# reshape for CrossEntropy
logits = outputs.view(-1, NUM_CHAR_CLASSES)
targets = labels.view(-1)

print(f"logits shape:  {logits.shape}")
print(f"targets shape: {targets.shape}")

loss = criterion(logits, targets)

print(f"\nLoss value: {loss.item()}")

### Predicition decoding test

In [ ]:
# =========================
# Prediction decoding test
# =========================

preds = outputs.argmax(dim=-1)

true_string = "".join(idx_to_char[int(i)] for i in labels[0])
pred_string = "".join(idx_to_char[int(i)] for i in preds[0])

print("\nExample prediction:")
print(f"True: {true_string}")
print(f"Pred: {pred_string}")

### Training batch predictions

In [ ]:
# =========================
# Debug: Check train batch predictions
# =========================

# Use when:
# - verifying the model can learn the training data
# - checking if the model is memorizing or completely failing
# - debugging low training accuracy

sample_logits = outputs[0].detach().cpu()  # [5, 51]

for pos in range(LABEL_LENGTH):
    probs = torch.softmax(sample_logits[pos], dim=0)
    top_probs, top_idxs = torch.topk(probs, k=5)
    top_chars = [idx_to_char[int(i)] for i in top_idxs]
    print(f"Position {pos}:")
    for ch, p in zip(top_chars, top_probs):
        print(f"  {ch}: {p.item():.4f}")

### Validation prediction diversity by position

In [ ]:
# =========================
# Debug: Character diversity per position
# =========================

# Use when:
# - predictions collapse to the same character
# - accuracy is very low
# - checking whether the model explores multiple characters

model.eval()

with torch.no_grad():
    train_batch = next(iter(train_loader))
    train_images, train_labels, _ = unpack_batch(train_batch)
    train_images = train_images.to(DEVICE)
    train_labels = train_labels.to(DEVICE)

    train_outputs = model(train_images)
    train_preds = train_outputs.argmax(dim=-1)

print("TRAIN BATCH")
for i in range(5):
    true_string = "".join(idx_to_char[int(x)] for x in train_labels[i])
    pred_string = "".join(idx_to_char[int(x)] for x in train_preds[i])
    print(f"True: {true_string} | Pred: {pred_string}")

### Propabilities per character position

In [ ]:
# =========================
# Debug: Top-k probabilities per position
# =========================

# Use when:
# - model predictions look wrong
# - inspecting model confidence
# - diagnosing confusion between characters

model.eval()

with torch.no_grad():
    batch = next(iter(val_loader))
    images, labels, _ = unpack_batch(batch)

    images = images.to(DEVICE)
    outputs = model(images)
    preds = outputs.argmax(dim=-1).cpu()

for pos in range(LABEL_LENGTH):
    unique_preds = torch.unique(preds[:, pos])
    decoded = [idx_to_char[int(x)] for x in unique_preds]
    print(f"Position {pos}: {decoded}")

### Character Confusion Matrix

In [ ]:
# =========================
# Debug: Character confusion matrix
# Use when:
# - character accuracy is low
# - some characters seem to be confused often
# - you want a visual summary of prediction errors
# =========================

from collections import Counter
import numpy as np

all_true_chars = []
all_pred_chars = []

model.eval()

with torch.no_grad():
    for batch in val_loader:
        images, labels, _ = unpack_batch(batch)
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        preds = outputs.argmax(dim=-1)

        labels = labels.cpu()
        preds = preds.cpu()

        all_true_chars.extend(labels.view(-1).tolist())
        all_pred_chars.extend(preds.view(-1).tolist())

num_classes = len(idx_to_char)
cm = np.zeros((num_classes, num_classes), dtype=int)

for t, p in zip(all_true_chars, all_pred_chars):
    cm[t, p] += 1

plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation="nearest", aspect="auto")
plt.title("Character Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.colorbar()
plt.tight_layout()
plt.show()

#### Better readable version for a subset of most-used characters

In [ ]:
# Show the 15 most frequent true characters
char_counts = Counter(all_true_chars)
top_chars_idx = [idx for idx, _ in char_counts.most_common(15)]

small_cm = cm[np.ix_(top_chars_idx, top_chars_idx)]
tick_labels = [idx_to_char[i] for i in top_chars_idx]

plt.figure(figsize=(8, 6))
plt.imshow(small_cm, interpolation="nearest", aspect="auto")
plt.title("Confusion Matrix (Top 15 Characters)")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks(range(len(tick_labels)), tick_labels, rotation=45)
plt.yticks(range(len(tick_labels)), tick_labels)
plt.colorbar()
plt.tight_layout()
plt.show()